### Re-Ranking Therotical concept

After the Retrival stage the re-ranking of the reponse by giving prompt as : Re- rank the order based on the query and Ranking the Response then 

In [2]:
from langchain.document_loaders import TextLoader
from langchain.embeddings import HuggingFaceBgeEmbeddings
from langchain.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_core.output_parsers import StrOutputParser

In [7]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load file
loader = TextLoader(r"D:\RAG\RAG_CHAIN\RAG-Optimization-Techniques\Sample_file.txt",encoding="utf-8")
raw_docs = loader.load()

# Split text into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(raw_docs)

docs

[Document(metadata={'source': 'D:\\RAG\\RAG_CHAIN\\RAG-Optimization-Techniques\\Sample_file.txt'}, page_content='. Introduction to LangChain\nLangChain is an open-source framework designed to simplify the development of applications that use large language models (LLMs). Instead of directly calling an LLM for every task, LangChain helps developers structure prompts, manage memory, connect external tools, and orchestrate workflows. This makes it particularly useful for building chatbots, question-answering systems, data pipelines, and AI agents.'),
 Document(metadata={'source': 'D:\\RAG\\RAG_CHAIN\\RAG-Optimization-Techniques\\Sample_file.txt'}, page_content='2. Core Philosophy\nThe key idea behind LangChain is that language models become much more powerful when they are chained with other components. A single prompt might generate an answer, but a chain of prompts, external API calls, databases, and reasoning steps can create intelligent systems. This “chaining” philosophy is what give

In [8]:
### user query 
query ="How can i use Langchain to build an application with memory and tools?"

In [9]:
#### FAISS and huggingface Embeddings 

embeddings = HuggingFaceBgeEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(docs,embeddings)
retriever=vectorstore.as_retriever(search_kwargs={"k":81})

C:\Users\siddv\AppData\Local\Temp\ipykernel_15504\584264190.py:3: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(model_name="all-MiniLM-L6-v2")
d:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceBgeEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000013057B61550>, search_kwargs={'k': 81})

In [14]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq  # assuming you're using LangChain Groq integration

# Load .env file
load_dotenv()

# Set API key in environment
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Initialize LLM
llm = ChatGroq(model="gemma2-9b-it")
print(llm)






client=<groq.resources.chat.completions.Completions object at 0x0000013032B87190> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001306144C950> model_name='gemma2-9b-it' model_kwargs={} groq_api_key=SecretStr('**********')


In [15]:
### Prompt template
Prompt=PromptTemplate.from_template("""
                                    you are a helpful assistant.your tast is to rank the following documents from the most to least relevant 
                                    user question ;"question"
                                    Documents:{documents}
                                    
                                    intructions : 
                                    - Think about the relevance of each document to the user's question.
                                    -Return a list of documnt Indices in ranked order , strating from the most relevant.
                                    
                                    Output format :Comma-separated document indoces(e.g,2,1,3,0)""")

In [16]:
retrived_docs= retriever.invoke(query)
retrived_docs


[Document(id='bcffac56-b24d-4791-8210-f6c5592660b8', metadata={'source': 'D:\\RAG\\RAG_CHAIN\\RAG-Optimization-Techniques\\Sample_file.txt'}, page_content='. Introduction to LangChain\nLangChain is an open-source framework designed to simplify the development of applications that use large language models (LLMs). Instead of directly calling an LLM for every task, LangChain helps developers structure prompts, manage memory, connect external tools, and orchestrate workflows. This makes it particularly useful for building chatbots, question-answering systems, data pipelines, and AI agents.'),
 Document(id='b062372b-ef44-4b71-9080-49661c110485', metadata={'source': 'D:\\RAG\\RAG_CHAIN\\RAG-Optimization-Techniques\\Sample_file.txt'}, page_content='3. Building Blocks\nLangChain provides several building blocks to create applications: prompts, chains, agents, and memory. Prompts define how instructions are given to the model. Chains connect multiple prompts and logic into workflows. Agents al

In [17]:
chain = Prompt | llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents'], input_types={}, partial_variables={}, template='\n                                    you are a helpful assistant.your tast is to rank the following documents from the most to least relevant \n                                    user question ;"question"\n                                    Documents:{documents}\n\n                                    intructions : \n                                    - Think about the relevance of each document to the user\'s question.\n                                    -Return a list of documnt Indices in ranked order , strating from the most relevant.\n\n                                    Output format :Comma-separated document indoces(e.g,2,1,3,0)')
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000013032B87190>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001306144C950>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr(

In [19]:
doc_lines=[f"{i+1}.{doc.page_content}" for i , doc in enumerate(retrived_docs)]
formatted_docs = "\n".join(doc_lines)
formatted_docs

'1.. Introduction to LangChain\nLangChain is an open-source framework designed to simplify the development of applications that use large language models (LLMs). Instead of directly calling an LLM for every task, LangChain helps developers structure prompts, manage memory, connect external tools, and orchestrate workflows. This makes it particularly useful for building chatbots, question-answering systems, data pipelines, and AI agents.\n2.3. Building Blocks\nLangChain provides several building blocks to create applications: prompts, chains, agents, and memory. Prompts define how instructions are given to the model. Chains connect multiple prompts and logic into workflows. Agents allow the model to decide which tools to use dynamically. Memory enables the system to remember past interactions, which is essential for conversational experiences.\n3.10. Conclusion\nLangChain is not just a framework but an enabler of the next generation of intelligent applications. By offering a structured 

In [20]:
response = chain.invoke({"question":query,"documents":formatted_docs})
response

"Here's a ranking of the documents based on relevance to a general question about LangChain:\n\n**7,2,1,4,5,8,9,6,3,10** \n\n**Explanation:**\n\n* **7 (Core Philosophy):**  Directly addresses the foundational concepts of LangChain, making it highly relevant.\n* **2 (Building Blocks):** Explains the essential components of LangChain, crucial for understanding its functionality.\n* **1 (Introduction):** Provides a general overview of LangChain, setting the stage for deeper understanding.\n* **4 (Integration with Tools and APIs):**  Highlights a key strength of LangChain, making it relevant to users interested in its practical applications.\n* **5 (Use Cases):** Shows real-world examples of LangChain in action, increasing relevance for those seeking practical applications.\n* **8 (Prompt Management):**  Focuses on a specific but important aspect of LangChain, relevant for developers.\n* **9 (Ecosystem and Community):**  Demonstrates the support and growth surrounding LangChain, valuable i

In [22]:
## Step 5. parse and rerank

indices =[int(x.strip())-1 for x in response.split(",") if x.strip().isdigit()]
indices

[1, 0, 3, 4, 7, 8, 5, 2]

In [23]:
renranked_docs = [retrived_docs[i] for i in indices if 0<=i < len(retrived_docs)]
renranked_docs

[Document(id='b062372b-ef44-4b71-9080-49661c110485', metadata={'source': 'D:\\RAG\\RAG_CHAIN\\RAG-Optimization-Techniques\\Sample_file.txt'}, page_content='3. Building Blocks\nLangChain provides several building blocks to create applications: prompts, chains, agents, and memory. Prompts define how instructions are given to the model. Chains connect multiple prompts and logic into workflows. Agents allow the model to decide which tools to use dynamically. Memory enables the system to remember past interactions, which is essential for conversational experiences.'),
 Document(id='bcffac56-b24d-4791-8210-f6c5592660b8', metadata={'source': 'D:\\RAG\\RAG_CHAIN\\RAG-Optimization-Techniques\\Sample_file.txt'}, page_content='. Introduction to LangChain\nLangChain is an open-source framework designed to simplify the development of applications that use large language models (LLMs). Instead of directly calling an LLM for every task, LangChain helps developers structure prompts, manage memory, con